# Lektion 13 - Agentminne


## Installation

Den här anteckningsboken visar hur man bygger en resebokningsagent med **persistent minne** med hjälp av **Microsoft Agent Framework** (MAF).

Du kommer att lära dig hur olika typer av agentminne — arbetsminne, korttids- och långtidsminne — påverkar hur en agent behåller och använder information över konversationer.

**Förutsättningar:**
- Ett Microsoft Foundry-projekt med en distribuerad chattmodell (t.ex. `gpt-4.1-mini`).
- Inloggad med Azure CLI — kör `az login` i din terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — din Microsoft Foundry-projekts slutpunkt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — namnet på din distribuerade modell.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Typer av agentminne

AI-agenter kan utnyttja olika typer av minne, var och en med ett särskilt syfte:

### Arbetsminne
Själva konversationskedjan — meddelandena som utbyts i en enda session. Agenten kan referera tillbaka till tidigare meddelanden i samma tråd för att upprätthålla sammanhang. I MAF skapas detta via **`agent.create_session()`**, vilket returnerar en `AgentSession`.

### Korttidsminne
Information som finns kvar under en uppgifts- eller sessions varaktighet men inte lagras permanent. Till exempel kan agenten samla fakta under en fleromgångsplaneringskonversation och använda dessa för att skapa en slutlig resplan.

### Långtidsminne
Preferenser och fakta som består **över flera sessioner**. En återvändande användare ska inte behöva upprepa sina kostrestriktioner eller resestil. Långtidsminne stöds vanligtvis av en extern lagring — en databas, fil eller vektorindex — och görs tillgängligt för agenten via verktyg.


## Arbetsminne med sessioner

Den enklaste formen av minne är konversationssessionen. När du skickar samma session (skapad via `agent.create_session()`) till på varandra följande `agent.run()` anrop, ser agenten hela historiken för den konversationen och kan komma ihåg tidigare detaljer.

Låt oss skapa en reseagent och demonstrera arbetsminne.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agenten kom ihåg budgeten korrekt eftersom båda meddelandena delar samma session. Detta är **arbetsminne** — det existerar endast under sessionens livstid.

### Vad händer med en ny tråd?

Om vi skapar en **ny** session har agenten inget minne av det föregående samtalet:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Långtidsminnesmönster

För att komma ihåg användarpreferenser **över sessioner** behöver vi en ihållande lagring som finns utanför konversations-tråden. Agenten får åtkomst till denna lagring genom **verktyg** — funktioner den kan anropa för att spara och hämta information.

Nedan implementerar vi en enkel preferenslagring i minnet (i produktion skulle du lagra detta i en databas eller vektorindex) och exponerar den som verktyg som agenten kan använda.

### Arkitektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Förstagångsanvändare bokar en jubileumsresa

Sarah besöker för första gången. Agenten bör lagra hennes preferenser via verktygen och använda dem för att rekommendera hotell.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah återvänder veckor senare

Sarah startar en **helt ny tråd** (simulerar en ny session). Arbetsminnet är tomt, men den långsiktiga preferenslagringen har fortfarande hennes information. Agenten bör hämta den och använda den för att anpassa rekommendationer.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sammanfattning

I den här lektionen lärde du dig tre typer av agentminne och hur man implementerar dem med Microsoft Agent Framework:

| Minnestyp | MAF-mekanism | Livslängd |
|---|---|---|
| **Arbetsminne** | `agent.create_session()` | Enskild konversation |
| **Korttidsminne** | Ackumulerad kontext inom en tråd | Enskild uppgift / session |
| **Långtidsminne** | Extern lagring som nås via `@tool`-funktioner | Över flera sessioner |

### Viktiga lärdomar
1. **`agent.create_session()`** tillhandahåller arbetsminne – agenten ser hela konversationshistoriken inom en session.
2. **Nya sessioner förlorar kontext** – utan långtidsminne kan inte agenten minnas tidigare konversationer.
3. **`@tool`-funktioner** överbryggar gapet – de låter agenten spara och hämta information från en beständig lagring.
4. **Personalisering förbättras över tid** – ju fler preferenser som lagras, desto bättre rekommendationer från agenten.

### Verkliga tillämpningar
- **Kundservice**: Kom ihåg kundhistorik och preferenser
- **Personliga assistenter**: Behåll kontext över dagar eller veckor
- **Hälso- och sjukvård**: Spåra patientinformation och preferenser
- **E-handel**: Personlig shopping baserat på historik

### Nästa steg
- Byt ut det minnesbaserade dict mot en databas eller vektorbutik (t.ex. Azure AI Search)
- Lägg till minnesutgång för tidskänslig information
- Bygg multi-agent system med delat minne
- Utforska Cognee-notebooken för kunskapsgrafstödd lagring


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
